# QLoRA Fine-Tuning — Primary Model (Colab)

**AAI-590 Capstone** · Team: Evin Joy, Sabina George, Jagadeesh Kumar Sellapan

Fine-tunes a compact open-weight LLM for medical question answering using
**QLoRA** (4-bit quantised base + trainable low-rank adapters). This is the
project's *primary* model; the from-scratch Transformer is the baseline.

**Requirements**
- A **GPU runtime** (Colab: *Runtime → Change runtime type → T4 GPU* or better).
- If the base model is gated (Mistral / Llama), a Hugging Face access token with
  the model's license accepted.

**Pipeline:** install deps → load & format MedMCQA + MedQA → load base model in
4-bit + LoRA → SFT training → accuracy check → save adapter.

### 1&nbsp;&nbsp;Install dependencies

In [1]:
# Colab GPU runtime required:  Runtime > Change runtime type > T4 GPU (or better)
!pip install -q -U peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 60.8 MB/s eta 0:00:00


### 2&nbsp;&nbsp;Imports and GPU check

In [2]:
import os, re, random
import torch
from datasets import load_dataset, Dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, TrainingArguments)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

assert torch.cuda.is_available(), \
    "No GPU detected. In Colab: Runtime > Change runtime type > GPU."
print("GPU:", torch.cuda.get_device_name(0))
random.seed(42); torch.manual_seed(42)

GPU: NVIDIA A100-SXM4-40GB


### 3&nbsp;&nbsp;Configuration

In [3]:
# --- Configuration --------------------------------------------------------- #
CFG = dict(
    # Primary model: a compact open-weight LLM. Mistral is *gated* on the Hub,
    # so accept its license and set an access token (next cell), OR switch to an
    # ungated model such as "Qwen/Qwen2.5-7B-Instruct" by editing model_name.
    model_name   = "mistralai/Mistral-7B-Instruct-v0.2",
    n_train      = 8000,     # training questions (raise once the pipeline works)
    n_val        = 500,      # held-out questions for the accuracy check
    max_seq_len  = 768,      # covers question + 4 options comfortably (see EDA)
    # QLoRA / LoRA adapter settings
    lora_r       = 16,
    lora_alpha   = 32,
    lora_dropout = 0.05,
    # Optimisation
    learning_rate = 2e-4,
    epochs        = 1,
    batch_size    = 2,       # per-device; effective batch = batch_size * grad_accum
    grad_accum    = 8,
    output_dir    = "qlora-medqa-adapter",
)

### 4&nbsp;&nbsp;Hugging Face access token (for gated models)

In [4]:
# If the chosen base model is gated (Mistral, Llama, ...), authenticate once.
# Get a token at https://huggingface.co/settings/tokens and paste it below,
# or run `from huggingface_hub import login; login()` and enter it interactively.
# os.environ["HF_TOKEN"] = "hf_xxx"   # <-- uncomment and set, or use login()

### 5&nbsp;&nbsp;Data loading and instruction formatting

In [5]:
# --- Load and format the medical MCQ data --------------------------------- #
# We pull MedMCQA and MedQA directly from the Hub and format each item as an
# instruction. The option order is shuffled (and the gold label remapped) to
# neutralise the positional bias identified during exploratory analysis.
LETTERS = "ABCD"

PROMPT = (
    "You are a medical expert answering a multiple-choice question. "
    "Select the single best option.\n\n"
    "Question: {q}\n"
    "A. {a}\nB. {b}\nC. {c}\nD. {d}\n\n"
    "Answer:"
)

def _records(n):
    """Yield up to n unified {question, options, answer_idx} records."""
    out = []
    medmcqa = load_dataset("openlifescienceai/medmcqa", split="train")
    for r in medmcqa:
        opts = [r["opa"], r["opb"], r["opc"], r["opd"]]
        if all(o for o in opts) and r["cop"] in (0, 1, 2, 3):
            out.append({"question": r["question"].strip(), "options": opts,
                        "answer_idx": int(r["cop"])})
        if len(out) >= n:
            return out
    medqa = load_dataset("GBaker/MedQA-USMLE-4-options", split="train")
    for r in medqa:
        o = r["options"]
        opts = [o["A"], o["B"], o["C"], o["D"]]
        idx = "ABCD".index(r["answer_idx"])
        out.append({"question": r["question"].strip(), "options": opts, "answer_idx": idx})
        if len(out) >= n:
            break
    return out

def _format(rec, with_answer=True, shuffle=True):
    opts, idx = list(rec["options"]), rec["answer_idx"]
    if shuffle:
        order = list(range(4)); random.shuffle(order)
        opts = [rec["options"][i] for i in order]
        idx = order.index(rec["answer_idx"])
    text = PROMPT.format(q=rec["question"], a=opts[0], b=opts[1], c=opts[2], d=opts[3])
    if with_answer:
        text = f"{text} {LETTERS[idx]}. {opts[idx]}"
    return text, idx

def build_datasets(cfg, eos_token):
    recs = _records(cfg["n_train"] + cfg["n_val"])
    random.shuffle(recs)
    train_recs, val_recs = recs[:cfg["n_train"]], recs[cfg["n_train"]:cfg["n_train"] + cfg["n_val"]]
    # Training text ends with the answer and an EOS token so the model learns to stop.
    train_texts = [_format(r, with_answer=True)[0] + eos_token for r in train_recs]
    train_ds = Dataset.from_dict({"text": train_texts})
    return train_ds, val_recs

### 6&nbsp;&nbsp;Load the base model in 4-bit and attach LoRA

In [6]:
# --- Load the base model in 4-bit and attach LoRA adapters ---------------- #
# QLoRA: the pretrained weights are quantised to 4-bit NormalFloat and frozen;
# only the injected low-rank adapters are trained.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,   # use torch.bfloat16 on A100/L4
)

tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    CFG["model_name"], quantization_config=bnb_config,
    device_map="auto", torch_dtype=torch.float16,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=CFG["lora_dropout"],
    bias="none", task_type="CAUSAL_LM",
    # Adapt all attention and MLP projections (standard for Llama/Mistral/Qwen).
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758


### 7&nbsp;&nbsp;Build the training set

In [7]:
train_ds, val_recs = build_datasets(CFG, tokenizer.eos_token)
print(f"Train examples: {len(train_ds):,} | Val examples: {len(val_recs):,}")
print("\n--- Example training text ---\n")
print(train_ds[0]["text"][:600])

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 85.9MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  936kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/182822 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6150 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4183 [00:00<?, ? examples/s]

Train examples: 8,000 | Val examples: 500

--- Example training text ---

You are a medical expert answering a multiple-choice question. Select the single best option.

Question: Colorado tick fever is the only known human infection caused by
A. Corona virus
B. Rotavirus
C. Coltivirus
D. Reovirus

Answer: C. Coltivirus</s>


### 8&nbsp;&nbsp;Fine-tune

In [8]:
from transformers import Trainer, DataCollatorForLanguageModeling

# Tokenise the formatted text; the collator builds causal-LM labels and pads.
def tokenize_fn(ex):
    return tokenizer(ex["text"], truncation=True, max_length=CFG["max_seq_len"])

tokenized = train_ds.map(tokenize_fn, remove_columns=["text"])
collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

train_args = TrainingArguments(
    output_dir=CFG["output_dir"],
    per_device_train_batch_size=CFG["batch_size"],
    gradient_accumulation_steps=CFG["grad_accum"],
    learning_rate=CFG["learning_rate"],
    num_train_epochs=CFG["epochs"],
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tokenized,
    data_collator=collator,
)
trainer.train()

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
10,1.584476
20,1.132541
30,1.041432
40,1.009631
50,1.038076
60,0.966449
70,0.993816
80,0.987364
90,0.925796
100,0.980839


TrainOutput(global_step=500, training_loss=0.9690670776367187, metrics={'train_runtime': 2797.5387, 'train_samples_per_second': 2.86, 'train_steps_per_second': 0.179, 'total_flos': 3.580226916601037e+16, 'train_loss': 0.9690670776367187, 'epoch': 1.0})

### 9&nbsp;&nbsp;Evaluate (held-out accuracy)

In [9]:
# --- Quick accuracy check on held-out questions --------------------------- #
# Greedy-decode a single answer letter per question and compare to the gold key.
model.eval()

@torch.no_grad()
def predict_letter(rec):
    prompt, gold_idx = _format(rec, with_answer=False, shuffle=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=4, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    gen = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    m = re.search(r"[ABCD]", gen.upper())
    pred_idx = "ABCD".index(m.group(0)) if m else 0
    return pred_idx, gold_idx

correct = sum(int(p == g) for p, g in (predict_letter(r) for r in val_recs))
acc = correct / len(val_recs)
print(f"Fine-tuned validation accuracy: {acc:.3f} on {len(val_recs)} questions "
      f"(random chance = 0.25)")

Fine-tuned validation accuracy: 0.608 on 500 questions (random chance = 0.25)


### 10&nbsp;&nbsp;Save the adapter

In [10]:
# --- Save the trained adapter --------------------------------------------- #
# Only the small LoRA adapter is saved (a few tens of MB), not the full base model.
model.save_pretrained(CFG["output_dir"])
tokenizer.save_pretrained(CFG["output_dir"])
print("Saved adapter to:", CFG["output_dir"])

# To download from Colab:
# import shutil; shutil.make_archive(CFG["output_dir"], "zip", CFG["output_dir"])
# from google.colab import files; files.download(CFG["output_dir"] + ".zip")

# To reload later for inference:
#   from peft import PeftModel
#   base = AutoModelForCausalLM.from_pretrained(CFG["model_name"], quantization_config=bnb_config, device_map="auto")
#   model = PeftModel.from_pretrained(base, CFG["output_dir"])

Saved adapter to: qlora-medqa-adapter


In [11]:
import shutil; shutil.make_archive("qlora-medqa-adapter", "zip", "qlora-medqa-adapter")
from google.colab import files; files.download("qlora-medqa-adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import torch, re

@torch.no_grad()
def base_predict(rec):
    prompt, gold = _format(rec, with_answer=False, shuffle=False)
    inp = tokenizer(prompt, return_tensors="pt").to(base_model.device)
    out = base_model.generate(**inp, max_new_tokens=4, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    gen = tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
    m = re.search(r"[ABCD]", gen.upper())
    return ("ABCD".index(m.group(0)) if m else 0), gold

base_model.eval()
correct = sum(int(p == g) for p, g in (base_predict(r) for r in val_recs))
print(f"Base (zero-shot) accuracy: {correct/len(val_recs):.3f} "
      f"on {len(val_recs)} questions (chance 0.25, fine-tuned was 0.608)")

Base (zero-shot) accuracy: 0.554 on 500 questions (chance 0.25, fine-tuned was 0.608)


In [14]:
import torch, gc, json, re
from transformers import (AutoModelForCausalLM, Trainer, TrainingArguments,
                          DataCollatorForLanguageModeling)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

SWEEP_N_TRAIN = 4000
sweep_train = train_ds.select(range(min(SWEEP_N_TRAIN, len(train_ds))))
tok_sweep = sweep_train.map(
    lambda ex: tokenizer(ex["text"], truncation=True, max_length=CFG["max_seq_len"]),
    remove_columns=["text"])
collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

def run_one(rank, alpha):
    base = AutoModelForCausalLM.from_pretrained(
        CFG["model_name"], quantization_config=bnb_config,
        device_map="auto", torch_dtype=torch.float16)
    base.config.use_cache = False
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    lcfg = LoraConfig(r=rank, lora_alpha=alpha, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
    m = get_peft_model(base, lcfg)
    args = TrainingArguments(output_dir=f"sweep_r{rank}", save_strategy="no",
        per_device_train_batch_size=CFG["batch_size"], gradient_accumulation_steps=CFG["grad_accum"],
        learning_rate=CFG["learning_rate"], num_train_epochs=1, warmup_ratio=0.03,
        lr_scheduler_type="cosine", logging_steps=25, fp16=True, optim="paged_adamw_8bit",
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        report_to="none")
    Trainer(model=m, args=args, train_dataset=tok_sweep, data_collator=collator).train()
    m.eval(); correct = 0
    for rec in val_recs:
        prompt, gold = _format(rec, with_answer=False, shuffle=False)
        inp = tokenizer(prompt, return_tensors="pt").to(m.device)
        with torch.no_grad():
            out = m.generate(**inp, max_new_tokens=4, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
        gen = tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
        mt = re.search(r"[ABCD]", gen.upper())
        correct += int(("ABCD".index(mt.group(0)) if mt else 0) == gold)
    acc = correct / len(val_recs)
    del m, base; gc.collect(); torch.cuda.empty_cache()
    return acc

results = {}
for r in [8, 16, 32]:
    acc = run_one(r, 2 * r)
    results[f"rank_{r}_alpha_{2*r}"] = round(acc, 4)
    print(f">>> LoRA rank {r}, alpha {2*r}: val accuracy = {acc:.3f}")

print("\nSweep results:", json.dumps(results, indent=2))
with open("sweep_results.json", "w") as f:
    json.dump(results, f, indent=2)
from google.colab import files; files.download("sweep_results.json")

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
25,1.275054
50,0.986677
75,0.965038
100,0.971799
125,0.965223
150,0.945200
175,0.944624
200,0.947073
225,0.957417
250,0.960317


>>> LoRA rank 8, alpha 16: val accuracy = 0.592


...

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
25,1.183049
50,0.992766
75,0.988941
100,0.992909
125,0.990587
150,0.960795
175,0.957938
200,0.957073
225,0.963621
250,0.965071


>>> LoRA rank 32, alpha 64: val accuracy = 0.576

Sweep results: {
  "rank_8_alpha_16": 0.592,
  "rank_16_alpha_32": 0.6,
  "rank_32_alpha_64": 0.576
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>